In [1]:
#Compas
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.model_selection import train_test_split

from csse import CSSE
from prepare_dataset import *

# Read Dataset Compas
df = prepare_compas_dataset("compas-scores-two-years.csv", "./data/")

#Get the input features
columns = df.columns.tolist()
columns = columns[-1:] + columns[:-1]
df = df[columns]
class_name = 'class' # class = High = "Bad class" / class = Medium-Low = "Good class"
columns_tmp = list(columns)
columns_tmp.remove(class_name)

x_train, x_test, y_train, y_test = train_test_split(df[columns_tmp], df[class_name], test_size=0.1)

model = RandomForestClassifier(n_estimators = 120, n_jobs=-1, random_state=0)  
model.fit(x_train, y_train)

p = model.predict(x_test)

print(classification_report(y_test, p))

              precision    recall  f1-score   support

        High       0.68      0.53      0.60       215
  Medium-Low       0.82      0.90      0.86       507

    accuracy                           0.79       722
   macro avg       0.75      0.72      0.73       722
weighted avg       0.78      0.79      0.78       722



In [2]:
#Armazenamento os dados processados
df.to_excel('dados.xlsx')

In [7]:
#-------Begin Parameter Adjustment--------
one_hot_lists = [ 
                  ['race_African-American', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 
                   'race_Native American', 'race_Other']
                ]

X = 0 #Indicates the instance's position to be explained in the dataset

Protected_feature = ['race_African-American', 'race_Asian', 'race_Caucasian', 'race_Hispanic',  
                     'race_Native American', 'race_Other']

#Se a coluna não foi codificada ONeHot basta passar o atributo escolhido como abaixo
#Protected_feature = 'sex'

#User preferences
#static_list = [] #List of features that cannot be changed. For example: static_list = ['age']
#K = 3 #Number of counterfactual explanations to be obtained

#Genetic Algorithm parameters
#num_gen = 30 #number of generations
#pop_size = 150 #population size
#per_elit = 0.1 #percentage of elitism
#cros_proba = 0.8 #crossover probability
#mutation_proba = 0.1 #mutation probability

#Weights of objective function metrics
#L1 = 1 #lambda 1 - Weight assigned the distance to the original instance
#L2 = 1 #lambda 2 - Weight assigned the amount of changes needed in the original instance

#copy the original instance
original_instance = x_test.iloc[X].copy()
      
#-------End Parameter Adjustment--------

print('Original instance - Class ' + str(p[X]) + '\n')
print(original_instance)
print('\nGetting counterfactuals...\n')

Original instance - Class Medium-Low

sex                         0
age                        32
age_cat                     1
priors_count                0
days_b_screening_arrest     1
c_charge_degree             0
is_recid                    0
is_violent_recid            0
two_year_recid              0
length_of_stay              0
race_African-American       1
race_Asian                  0
race_Caucasian              0
race_Hispanic               0
race_Native American        0
race_Other                  0
Name: 2753, dtype: int64

Getting counterfactuals...



In [8]:
#Indivíduo gerado para avaliar injustiça
test = original_instance.copy()

#Armazena a raça e classe original do indivíduo
original_race = original_instance[Protected_feature]
classe_original = p[X]

# Verifica qual categoria de raça está ativa
if test['race_African-American'] == 1:
    # African-American -> Native American
    contrafactual_race = 'race_Native American'
else:
    # Any other race -> African-American
    contrafactual_race = 'race_African-American'

#Gerando o I' do artigo
# Zera todas as categorias da raça
test[Protected_feature] = 0

# Ativa a nova categoria de raça
test[contrafactual_race] = 1

#Dataframe criado para testar a injustiça
df_teste_injustica = pd.DataFrame(columns=columns_tmp)
df_teste_injustica.loc[0] = original_instance.copy()
df_teste_injustica.loc[1] = test.copy() 
p2 = model.predict(df_teste_injustica)

#Verifica se a simples mudaça na raça foi suficiente para evidenciar a injustiça
if p2[0] != p2[1]:
    print('Injustiça detectada')
    print('Instância 1 - Class ' + str(classe_original) + '\n')##############
    print(original_instance)############

    #Imprimindo os indivíduos que evidenciam a injustiça
    print('Instância 2 - Class contrafactual \n')###########
    print(test)##########
else:
    #Run CSSE - Method executed with default parameters
    explainerCSSE = CSSE(df[columns_tmp], model, static_list = [Protected_feature], one_hot_lists = one_hot_lists)

    #Procura contrafactuais para a instância teste - os contrafactuais gerados serão o "C" do artigo 
    contrafactual_set, solution = explainerCSSE.explain(test, p2[1]) #Method returns the list of counterfactuals and the explanations generated from them

    #The method returns a list of counterfactual solutions, where each solution, in turn, is a change list (each change has the "column" and "value" to be changed). To implement another output format, see the "printResults" function
    #explainerCSSE.printResults(solution)

    # Volta os contrafactuais à raça original (procurando o indivíduo C')
    for i in range(0, len(contrafactual_set)):
        contrafactual_set.loc[i, Protected_feature] = original_race.values    
        
    #Dataframe criado para testar se voltando o individuo a raça original inverte a classe
    p3 = model.predict(contrafactual_set)

    #Variável para verificar se foi encontrado algum individuo que evidencia injustça
    injustica_detectada = False ############
    #Verifica se algum indíduo volta a classe original com a mudança de raça
    for j in range (0, len(p3)):
        if p3[j] == classe_original:
            injustica_detectada = True
            index_injustica = j
            print('\n Injustiça detectada') 

            #Imprimindo os indivíduos que evidenciam a injustiça
            print('Instância 1 - Class ' + str(classe_original) + '\n')
            print(contrafactual_set.iloc[j])

            individuo_injustica = contrafactual_set.iloc[j].copy()

            # Zera todas as categorias da raça
            individuo_injustica[Protected_feature] = 0

            # Ativa a raça contrafactual
            individuo_injustica[contrafactual_race] = 1

            print('Instância 2 - Class contrafactual \n')
            print(individuo_injustica)
    if not injustica_detectada:##########
        print('\n Injustiça não detectada')  ###########              

Processing...: 100%|███████████████████████████████████████████████████████████████████| 30/30 [00:40<00:00,  1.35s/it]


 Injustiça detectada
Instância 1 - Class Medium-Low

sex                         0
age                        32
age_cat                     1
priors_count                7
days_b_screening_arrest     1
c_charge_degree             0
is_recid                    0
is_violent_recid            0
two_year_recid              0
length_of_stay              0
race_African-American       1
race_Asian                  0
race_Caucasian              0
race_Hispanic               0
race_Native American        0
race_Other                  0
Name: 0, dtype: int64
Instância 2 - Class contrafactual 

sex                         0
age                        32
age_cat                     1
priors_count                7
days_b_screening_arrest     1
c_charge_degree             0
is_recid                    0
is_violent_recid            0
two_year_recid              0
length_of_stay              0
race_African-American       0
race_Asian                  0
race_Caucasian              0
race_Hispanic     